# Building Custom Agents - Nexus-LLM

This notebook shows how to build custom AI agents that can reason, plan, and use tools to accomplish complex tasks.

## What is an Agent?

An agent combines an LLM with a set of tools and a reasoning loop. The LLM decides which tools to call, processes results, and iterates until the task is complete.

## 1. Basic Agent Setup

In [ ]:
from nexus_llm import InferenceEngine
from nexus_llm.agents import Agent, Tool, ToolRegistry, AgentConfig

engine = InferenceEngine(model_name="nexus-7b-chat", device="auto")

# Define a simple tool
class CalculatorTool(Tool):
    name = "calculator"
    description = "Evaluates math expressions. Input: a Python math expression string."

    def run(self, expression: str) -> str:
        import json
        try:
            allowed = {"abs": abs, "round": round, "min": min, "max": max, "pow": pow}
            result = eval(expression, {"__builtins__": {}}, allowed)
            return json.dumps({"result": result})
        except Exception as e:
            return json.dumps({"error": str(e)})

# Register tools
registry = ToolRegistry()
registry.register(CalculatorTool())

# Create agent
agent = Agent(
    inference_engine=engine,
    tool_registry=registry,
    config=AgentConfig(max_iterations=5, verbose=True),
)

print("Agent created with tools:", [t.name for t in registry.list_tools()])

## 2. Running the Agent

In [ ]:
# Run a simple task
result = agent.run("What is 15 squared plus 27?")

print(f"Answer: {result.answer}")
print(f"Steps: {result.num_steps}")
print(f"Tools used: {result.tools_used}")

In [ ]:
# Run a multi-step task
result = agent.run("Calculate the area of a circle with radius 7, then divide by pi.")

print(f"Answer: {result.answer}")
print(f"Reasoning trace:")
for step in result.trace:
    if step.type == "thinking":
        print(f"  [Think] {step.content[:100]}...")
    elif step.type == "tool_call":
        print(f"  [Tool] {step.tool_name}({step.input})")
    elif step.type == "tool_result":
        print(f"  [Result] {step.output}")

## 3. Adding More Tools

In [ ]:
import json
import datetime

class DateTimeTool(Tool):
    name = "datetime"
    description = "Gets the current date and time. Input: timezone name (e.g., 'UTC', 'US/Eastern')."

    def run(self, timezone: str = "UTC") -> str:
        now = datetime.datetime.now()
        return json.dumps({"datetime": now.isoformat(), "timezone": timezone})

class ListTool(Tool):
    name = "list_operations"
    description = "Performs operations on lists: sum, mean, sort, reverse. Input: JSON with 'operation' and 'data' keys."

    def run(self, params: str) -> str:
        params = json.loads(params)
        operation = params["operation"]
        data = params["data"]

        if operation == "sum":
            return json.dumps({"result": sum(data)})
        elif operation == "mean":
            return json.dumps({"result": sum(data) / len(data)})
        elif operation == "sort":
            return json.dumps({"result": sorted(data)})
        elif operation == "reverse":
            return json.dumps({"result": list(reversed(data))})
        else:
            return json.dumps({"error": f"Unknown operation: {operation}"})

registry.register(DateTimeTool())
registry.register(ListTool())

print(f"Available tools: {[t.name for t in registry.list_tools()]}")

## 4. Complex Multi-Tool Task

In [ ]:
result = agent.run(
    "Get the current date, then calculate how many days are left in the year, "
    "and find the mean of the list [10, 20, 30, 40, 50]."
)

print(f"Answer: {result.answer}")
print(f"Tools used: {result.tools_used}")

## 5. Streaming Agent Output

In [ ]:
# Stream the agent's reasoning in real-time
print("Running agent with streaming...\n")

for step in agent.run_stream("Calculate the Fibonacci sequence up to the 10th term."):
    if step.type == "thinking":
        print(f"\n[Thinking] {step.content}")
    elif step.type == "tool_call":
        print(f"\n[Calling] {step.tool_name}({step.input})")
    elif step.type == "tool_result":
        print(f"[Result] {step.output}")
    elif step.type == "answer":
        print(f"\n[Final Answer] {step.content}")

## 6. Building a Specialized Agent

In [ ]:
# Create a data analysis agent with specialized system prompt
data_agent = Agent(
    inference_engine=engine,
    tool_registry=registry,
    config=AgentConfig(
        max_iterations=10,
        system_prompt="""You are a data analysis expert. When given a data task:
1. First, understand what analysis is needed
2. Plan your approach step by step
3. Use tools to perform calculations
4. Provide clear, well-formatted results
Always show your work and explain your reasoning.""",
    ),
)

result = data_agent.run(
    "Analyze this dataset: [23, 45, 67, 89, 12, 34, 56, 78]. "
    "Calculate the sum, mean, and sorted order."
)

print(f"Analysis Result:\n{result.answer}")

## Key Takeaways

- **Tools** define what the agent can do - give it the right tools for the job
- **System prompts** shape the agent's behavior and reasoning style
- **Max iterations** prevent infinite loops - set appropriately for your task
- **Streaming** provides real-time visibility into the agent's thought process
- **Trace logging** helps debug and optimize agent performance